In [0]:
from pyspark.sql import SparkSession 
spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()


In [0]:
data = [
     (1, "C001", "Laptop", "50000", "2024-01-01"),
    (2, "C002", "Mobile", None, "2024-01-02"),
  (3, "C003", "Tablet", "20000", "2024-01-03"), 
    (4, "C004", "Laptop", "55000", "2024-01-04"),
   (5, "C005", "Headphones", None, "2024-01-05"),   
   
   (6, "C006", "Camera", "30000", "2024-01-06"),
   (7, "C007", "Mobile", "18000", "2024-01-07"),
    (8, "C008", "Watch", "8000", "2024-01-07")]

In [0]:
columns = ["order_id", "customer_id", "product", "amount", "updated_date"]


In [0]:
 df = spark.createDataFrame(data, columns)

In [0]:
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,null,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,null,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
df=df.withColumn("amount", col("amount").cast(IntegerType()))
df.display()


order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,null,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,null,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
df=df.fillna({"amount":1000})

In [0]:
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
df=df.withColumn("updated_date",col("updated_date").cast("date"))
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
df=df.withColumn("bonus",col("amount")*0.1)
df.display()


order_id,customer_id,product,amount,updated_date,bonus
1,C001,Laptop,50000,2024-01-01,5000.0
2,C002,Mobile,1000,2024-01-02,100.0
3,C003,Tablet,20000,2024-01-03,2000.0
4,C004,Laptop,55000,2024-01-04,5500.0
5,C005,Headphones,1000,2024-01-05,100.0
6,C006,Camera,30000,2024-01-06,3000.0
7,C007,Mobile,18000,2024-01-07,1800.0
8,C008,Watch,8000,2024-01-07,800.0


In [0]:
from pyspark.sql.functions import when
df=df.withColumn("grade",when(col('amount') >  20000,"high").otherwise("low"))

In [0]:
df.display()

order_id,customer_id,product,amount,updated_date,bonus,grade
1,C001,Laptop,50000,2024-01-01,5000.0,high
2,C002,Mobile,1000,2024-01-02,100.0,low
3,C003,Tablet,20000,2024-01-03,2000.0,low
4,C004,Laptop,55000,2024-01-04,5500.0,high
5,C005,Headphones,1000,2024-01-05,100.0,low
6,C006,Camera,30000,2024-01-06,3000.0,high
7,C007,Mobile,18000,2024-01-07,1800.0,low
8,C008,Watch,8000,2024-01-07,800.0,low


In [0]:
def grades(amount):
    if amount > 20000:
        return "high"
    else:
        return "low"
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
udf_grade = udf(grades, StringType())
df=df.withColumn("grades",udf_grade(col("amount")))
df.display()

order_id,customer_id,product,amount,updated_date,bonus,grade,grades
1,C001,Laptop,50000,2024-01-01,5000.0,high,high
2,C002,Mobile,1000,2024-01-02,100.0,low,low
3,C003,Tablet,20000,2024-01-03,2000.0,low,low
4,C004,Laptop,55000,2024-01-04,5500.0,high,high
5,C005,Headphones,1000,2024-01-05,100.0,low,low
6,C006,Camera,30000,2024-01-06,3000.0,high,high
7,C007,Mobile,18000,2024-01-07,1800.0,low,low
8,C008,Watch,8000,2024-01-07,800.0,low,low


In [0]:
df.write.mode("overwrite").format("delta").save("/Volumes/workspace/default/end-end-pipeline")

df.display()

order_id,customer_id,product,amount,updated_date,bonus,grade,grades
1,C001,Laptop,50000,2024-01-01,5000.0,high,high
2,C002,Mobile,1000,2024-01-02,100.0,low,low
3,C003,Tablet,20000,2024-01-03,2000.0,low,low
4,C004,Laptop,55000,2024-01-04,5500.0,high,high
5,C005,Headphones,1000,2024-01-05,100.0,low,low
6,C006,Camera,30000,2024-01-06,3000.0,high,high
7,C007,Mobile,18000,2024-01-07,1800.0,low,low
8,C008,Watch,8000,2024-01-07,800.0,low,low


In [0]:
new_data = [
    (3, "C003", "Tablet", "22000", "2024-01-06"),  # updated
    (9, "C009", "Laptop", "60000", "2024-01-08")   # new
]

new_df = spark.createDataFrame(new_data, columns)

In [0]:
from pyspark.sql.functions import col

last_loaded_date = "2024-01-05"

incremental_df = df.filter(col("date") > last_loaded_date)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("id").orderBy(col("date").desc())

dedup_df = incremental_df.withColumn(
    "rank", row_number().over(window_spec)
).filter(col("rank") == 1).drop("rank")